# Example Pipeline

Este es el scarper base de ejemplo en el que se van a basar todos los scrapers.

En la siguiente sección se define el sube dos niveles en sistema para la detección de los paquetes auxiliares.

## Development Options Activation

In [1]:
IS_DEVELOPMENT = True

## Pipeline Setup

In [2]:
CLIENT_NAME = None
PIPELINE_NAME = "example_pipeline"

In [3]:
import os
import sys

if not IS_DEVELOPMENT:
    CLIENT_NAME = sys.argv[1]

else:
    CLIENT_NAME = "example_client"

# Subimos un nivel para llegar a 'scraper_pipelines' y lo añadimos al path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

## Libraries

In [4]:
import time
import polars as pl
from scrapers_utils import BaseScraper
from pipeline_utils import RunManager
from pipeline_utils.cleaning_utils import strip_all_str
from selenium.webdriver.common.by import By

## Pipeline Initialization

In [5]:
# Development options
if IS_DEVELOPMENT:
    headless = False
else:
    headless = True

In [6]:
# Run Manager Initialization
run_manager = RunManager(CLIENT_NAME, PIPELINE_NAME)

In [7]:
# Scraper initializiation
scraper = BaseScraper.create_with_decodo(port=20001, timeout=30, headless = headless)
scraper.driver.get("https://www.scrapethissite.com/")

Testing Decodo proxy connection...
Proxy Active! Current IP: None in {'name': 'Mexico', 'code': 'MX', 'continent': 'North America'}


In [8]:
run_manager.step = "DATA_RECOLLECTION"

# Data Recolection

## Home Page

### Elements:

In [9]:
SANDBOX_BUTTON = (By.XPATH, "//a[contains(text(), 'Explore Sandbox')]")

### Actions:

In [10]:
try:
    sandbox_button = scraper.wait_for_clickable(SANDBOX_BUTTON)
    sandbox_button.click()
except Exception as e:
    scraper.quit()
    run_manager.set_run_fail(f"SANDBOX_BUTTON: {e}")

In [11]:
scraper.human_jitter()

## Lists Page

### Elements:

In [12]:
HOCKEY_TEAMS_BUTTON = (By.XPATH, "//a[contains(text(), 'Hockey Teams')]")

### Actions:

In [13]:
try:
    hockey_teams_button = scraper.wait_for_clickable(HOCKEY_TEAMS_BUTTON)
    hockey_teams_button.click()
except Exception as e:
    scraper.quit()
    run_manager.set_run_fail(f"HOCKEY_TEAMS_BUTTON: {e}")

In [14]:
scraper.human_jitter()

## Hockey Teams Page

### Elements:

In [15]:
TABLE_HEADERS = (By.XPATH, "//table[@class='table']//th")
TEAMS_DATA = (By.XPATH, "//table[@class='table']//tr[@class='team']")
TEAM_ROW = (By.XPATH, ".//td")
NEXT_PAGE_BUTTON = (By.XPATH, "//a[@aria-label='Next']")


### Actions:

In [16]:
raw_data_dict = {}

In [17]:
# Raw data dict initialization
try:
    headers_elements_list = scraper.get_all_objects(TABLE_HEADERS)
    for header_elmt in headers_elements_list:
        raw_data_dict[header_elmt.text] = []
    headers = raw_data_dict.keys()
except Exception as e:
    scraper.quit()
    run_manager.set_run_fail(f"TABLE_HEADERS: {e}")

In [18]:
def get_table_data():
    teams_table_elements = scraper.get_all_objects(TEAMS_DATA)
    for team in teams_table_elements:
        row = team.find_elements(*TEAM_ROW)
        for i, key in enumerate(headers):
            raw_data_dict[key].append(row[i].text)
        

In [19]:
# Pagination and data recollection
try:
    while True:
        next_page_button = scraper.wait_for_present(NEXT_PAGE_BUTTON)
        if next_page_button is not None:
            next_page_button.click()
            get_table_data()
            scraper.human_jitter()
        else:
            break
except Exception as e:
    scraper.quit()
    run_manager.set_run_fail(f"NEXT_PAGE_BUTTON  or TEAMS_DATA: {e}")

In [20]:
# Closing scraper
scraper.quit()

## Storing raw data

In [21]:
try:
    run_manager.store_raw_data(raw_data_dict)
except Exception as e:
    run_manager.set_run_fail(f"STORING_RAW_DATA: {e}")

# Data Cleaning

In [22]:
run_manager.step = "CLEANING_RAW_DATA"

In [23]:
raw_df = pl.DataFrame(raw_data_dict)
clean_df = raw_df.clone()

In [24]:
# All str strip
clean_df = strip_all_str(clean_df)

In [25]:
# Specific Type Casting and Values Cleaning
clean_df = clean_df.with_columns([
    pl.col("Team Name").replace("","0").cast(pl.String),
    pl.col("Year").replace("","0").cast(pl.Int64),
    pl.col("Wins").replace("","0").cast(pl.Int64),
    pl.col("Losses").replace("","0").cast(pl.Int64),
    pl.col("OT Losses").replace("","0").cast(pl.Int64),
    pl.col("Win %").cast(pl.Float64),
    pl.col("Goals For (GF)").replace("","0").cast(pl.Int64),
    pl.col("Goals Against (GA)").replace("","0").cast(pl.Int64),
    pl.col("+ / -").replace("","0").cast(pl.Int64)
])

In [26]:
cleaned_data_dict = clean_df.to_dict(as_series=False)

### Storing Cleaned Data

In [27]:
try:
    run_manager.store_cleaned_data(raw_data_dict)
except Exception as e:
    run_manager.set_run_fail(f"STORING_CLEANED_DATA: {e}")

## Pipeline Finished